In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsgr18smartdata";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.stores (
store_nbr integer,
city string,
state string,
type string,
cluster integer
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/stores"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.train (
id integer,
date timestamp,
store_nbr integer,
family string,
sales double,
onpromotion integer
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/train"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.transactions (
 date timestamp,
 store_nbr integer,
  transactions integer
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/transactions"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.transformed (
store_nbr integer,
city string,
state string,
type string,
cluster integer,
id integer,
date timestamp,
family string,
sales integer,
onpromotion integer,
transactions integer
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/transformed"

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_partitioned (
  year integer,
  conteo long,
  max integer,
  min integer,
  state string,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_partitioned"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS adbsflordeoro.default.bronze_train (
    id STRING,
    date STRING,
    store_nbr STRING,
    family STRING,
    sales STRING,
    onpromotion STRING
)
USING DELTA
LOCATION 'abfss://bronze@adlsgr18smartdata.dfs.core.windows.net/bronze_train';

In [0]:
%sql
-- Tabla de ventas: train.csv
-- Ya existe y está ubicada en el contenedor Bronze.
CREATE TABLE IF NOT EXISTS adbsflordeoro.default.bronze_train (
    id STRING,
    date STRING,
    store_nbr STRING,
    family STRING,
    sales STRING,
    onpromotion STRING
)
USING DELTA
LOCATION 'abfss://bronze@adlsgr18smartdata.dfs.core.windows.net/bronze_train';


-- Nueva tabla externa para transactions.csv
CREATE TABLE IF NOT EXISTS adbsflordeoro.default.bronze_transactions_ext (
    date DATE,
    store_nbr INT,
    transactions INT,
    _source_file STRING,
    _ingestion_timestamp TIMESTAMP,
    _ingestion_date DATE
)
USING DELTA
LOCATION 'abfss://bronze@adlsgr18smartdata.dfs.core.windows.net/bronze_transactions_ext';


-- Nueva tabla externa para stores.csv
CREATE TABLE IF NOT EXISTS adbsflordeoro.default.bronze_stores_ext (
    store_nbr INT,
    city STRING,
    state STRING,
    type STRING,
    cluster INT,
    _source_file STRING,
    _ingestion_timestamp TIMESTAMP,
    _ingestion_date DATE
)
USING DELTA
LOCATION 'abfss://bronze@adlsgr18smartdata.dfs.core.windows.net/bronze_stores_ext';